In [ ]:
# 1. 라이브러리 및 데이터 로드 (Phase 12: Trackman 물리 특성 융합)
import pandas as pd
import numpy as np
import os
import time
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

DATA_DIR = r"C:\Users\이호준\OneDrive\바탕 화면\LG aimers\open\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./data" # 코랩 환경 대비

ID_COL = "row_id"
TARGET_COL = "control_success"

# 롤백 유지: 카테고리 폭발 오버피팅을 방지하기 위해 ID는 수치형으로 유지
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand"]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), encoding="utf-8-sig", nrows=0).columns

tm_stats = pd.read_csv(os.path.join(DATA_DIR, "pitcher_trackman_stats.csv"))
train = pd.merge(train, tm_stats, on='pitcher_id', how='left')

tm_features = ['tm_rel_speed', 'tm_spin_rate', 'tm_ivb', 'tm_hb', 'tm_extension']
FEATURES = [c for c in test_cols if c != ID_COL] + tm_features
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f"Trackman 융합 완료. 피처 수: {len(FEATURES)} (새로 추가된 물리 특성: {len(tm_features)}개)")


In [ ]:
# 2. 전처리 파이프라인 및 가장 방어력이 좋았던 모델 정의
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])

lgb_model = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.03, num_leaves=31, max_depth=6,
    min_child_samples=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=-1
)

xgb_model = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.03, max_depth=6,
    min_child_weight=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, tree_method='hist'
)

cb_model = cb.CatBoostClassifier(
    iterations=300, learning_rate=0.03, depth=6,
    min_data_in_leaf=500, random_seed=42, verbose=0
)


In [ ]:
# 3. 2024년 시계열 검증 및 수학적 확률 보정기(Isotonic) 추출
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET_COL]
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET_COL]

print("전처리 적용 중...")
preprocessor.fit(X_train)
X_train_pre = preprocessor.transform(X_train)
X_val_pre = preprocessor.transform(X_val)

print("1. LightGBM 학습...")
lgb_model.fit(X_train_pre, y_train)
lgb_preds = lgb_model.predict_proba(X_val_pre)[:, 1]

print("2. XGBoost 학습...")
xgb_model.fit(X_train_pre, y_train)
xgb_preds = xgb_model.predict_proba(X_val_pre)[:, 1]

print("3. CatBoost 학습...")
cb_model.fit(X_train_pre, y_train)
cb_preds = cb_model.predict_proba(X_val_pre)[:, 1]

# 839점 황금 밸런스 1/3 균등 앙상블
ensemble_preds = (lgb_preds + xgb_preds + cb_preds) / 3.0

print("\n🔧 수제 Isotonic 보정기 작동 중...")
iso = IsotonicRegression(out_of_bounds='clip')
calibrated_preds = iso.fit_transform(ensemble_preds, y_val)

r = y_val.mean()
brier = ((calibrated_preds - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"\n✅ Brier Score: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"🚀 트랙맨이 융합된 보정 Validation Score: {score:.2f} (1000점 폭발 대기 중!)")

iso_data = {
    "X_thresh": iso.X_thresholds_.tolist(),
    "y_thresh": iso.y_thresholds_.tolist()
}
os.makedirs("./submission/model", exist_ok=True)
with open("./submission/model/iso_thresholds.json", "w") as f:
    json.dump(iso_data, f)


In [ ]:
# 4. 전체 데이터 재학습 및 에러 없는 script.py 생성
import zipfile
import shutil

print("전체 데이터 전처리 및 3대장 재학습 중...")
preprocessor.fit(train[FEATURES])
X_all_pre = preprocessor.transform(train[FEATURES])
y_all = train[TARGET_COL]

lgb_model.fit(X_all_pre, y_all)
xgb_model.fit(X_all_pre, y_all)
cb_model.fit(X_all_pre, y_all)
print("학습 완료.")

joblib.dump(preprocessor, "./submission/model/preprocessor.pkl", compress=3)
lgb_model.booster_.save_model("./submission/model/lgb.txt")
xgb_model.save_model("./submission/model/xgb.json")
cb_model.save_model("./submission/model/cb.cbm")
joblib.dump(FEATURES, "./submission/model/features.pkl")

# 추론 서버에서 병합할 수 있도록 트랙맨 스탯 파일을 모델 폴더에 챙겨넣습니다.
shutil.copy(os.path.join(DATA_DIR, "pitcher_trackman_stats.csv"), "./submission/model/pitcher_trackman_stats.csv")

with open("./submission/requirements.txt", "w") as f:
    f.write("pandas\nscikit-learn\nlightgbm\nxgboost\ncatboost\njoblib\n")

script_code = """import os
import joblib
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

ID_COL = "row_id"
TARGET_COL = "control_success"

def merge_predictions(sub, ids, preds):
    pred_map = dict(zip(ids, preds))
    values = [pred_map.get(rid, cur) for rid, cur in zip(sub[ID_COL], sub[TARGET_COL])]
    sub[TARGET_COL] = values
    return sub

def main():
    TEST_PATH = "./data/test.csv"
    SAMPLE_SUB_PATH = "./data/sample_submission.csv"
    OUT_PATH = "./output/submission.csv"

    preprocessor = joblib.load("./model/preprocessor.pkl")
    FEATURES = joblib.load("./model/features.pkl")
    tm_stats = pd.read_csv("./model/pitcher_trackman_stats.csv")
    
    with open("./model/iso_thresholds.json", "r") as f:
        iso_data = json.load(f)
    X_thresh = np.array(iso_data["X_thresh"])
    y_thresh = np.array(iso_data["y_thresh"])
    
    lgb_booster = lgb.Booster(model_file="./model/lgb.txt")
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model("./model/xgb.json")
    cb_model = cb.CatBoostClassifier()
    cb_model.load_model("./model/cb.cbm")
    
    test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")
    sub = pd.read_csv(SAMPLE_SUB_PATH, encoding="utf-8-sig")
    
    test = pd.merge(test, tm_stats, on='pitcher_id', how='left')
    
    ids = test[ID_COL].tolist()
    X = test[FEATURES]
    X_pre = preprocessor.transform(X)
    
    lgb_preds = lgb_booster.predict(X_pre)
    xgb_preds = xgb_model.predict_proba(X_pre)[:, 1]
    cb_preds = cb_model.predict_proba(X_pre)[:, 1]
    raw_ensemble_preds = (lgb_preds + xgb_preds + cb_preds) / 3.0
    
    calibrated_preds = np.interp(raw_ensemble_preds, X_thresh, y_thresh)
    
    sub = merge_predictions(sub, ids, calibrated_preds)
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    sub.to_csv(OUT_PATH, index=False, encoding="utf-8")

if __name__ == "__main__":
    main()
"""

with open("./submission/script.py", "w", encoding="utf-8") as f:
    f.write(script_code)

zip_path = 'trackman_submit.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk('./submission'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, './submission'))

print("✅ Trackman 179만 줄의 스탯이 완벽하게 융합된 trackman_fusion_submit.zip 생성 완료!")
